In [0]:

%pip install yfinance pandas
     

In [0]:
import yfinance as yf
import pandas as pd
import os
from datetime import datetime

# ---------------------------------------------------
# Tickers (UK equities)
# ---------------------------------------------------
tickers = [
    "SHEL.L", "HSBA.L", "BP.L", "ULVR.L", "VOD.L",
    "BARC.L", "AZN.L", "SGE.L", "GRG.L", "BWY.L",
    "RSW.L", "PNN.L", "SVT.L", "IMI.L", "BBY.L",
    "DRX.L", "TEP.L", "CNA.L", "NCC.L", "SBRY.L"
]

# ---------------------------------------------------
# Company metadata (dimension)
# ---------------------------------------------------

ticker_meta = {
    "SHEL.L": "Shell plc",
    "HSBA.L": "HSBC Holdings plc",
    "BP.L": "BP p.l.c.",
    "ULVR.L": "Unilever PLC",
    "VOD.L": "Vodafone Group Plc",
    "BARC.L": "Barclays PLC",
    "AZN.L": "AstraZeneca PLC",
    "SGE.L": "The Sage Group plc",
    "GRG.L": "Greggs plc",
    "BWY.L": "Bellway plc",
    "RSW.L": "Renishaw plc",
    "PNN.L": "Pennon Group plc",
    "SVT.L": "Severn Trent plc",
    "IMI.L": "IMI plc",
    "BBY.L": "Balfour Beatty plc",
    "DRX.L": "Drax Group plc",
    "TEP.L": "Telecom Plus plc",
    "CNA.L": "Centrica plc",
    "NCC.L": "NCC Group plc",
    "SBRY.L": "J Sainsbury plc"
}

# ---------------------------------------------------
# Time partition (snapshot date)
# ---------------------------------------------------
now = datetime.now()
year = now.strftime("%Y")
month = now.strftime("%m")
day = now.strftime("%d")

BASE_PATH = "/Volumes/corporate_data_lakehouse/bronze/raw_comp_data/yfinance"

# ---------------------------------------------------
# Dataset folders
# ---------------------------------------------------
datasets = [
    "income_statement",
    "balance_sheet",
    "cashflow",
    "history",
    "stats"
]

# ---------------------------------------------------
# Create folder structure
# ---------------------------------------------------
for d in datasets:
    os.makedirs(f"{BASE_PATH}/{d}/{year}/{month}/{day}", exist_ok=True)

# ---------------------------------------------------
# Helper function to enrich dataframe
# ---------------------------------------------------
def enrich(df, ticker):
        df = df.copy()
        df["ticker"] = ticker
        df["company_name"] = ticker_meta.get(ticker, "Unknown")
        df["ingestion_ts"] = datetime.now()
        df["source"] = "yfinance"
        return df


# ---------------------------------------------------
# Ingestion loop
# ---------------------------------------------------

for t in tickers:
    print("Processing:", t)

    stock = yf.Ticker(t)

    # ---------------- Income Statement ----------------
    try:
        income = stock.financials.T
        income = enrich(income, t)
        income.to_json(
            f"{BASE_PATH}/income_statement/{year}/{month}/{day}/{t}.json",
            orient="index"
        )
    except Exception as e:
        print("Income error:", t, e)

    # ---------------- Balance Sheet ----------------
    try:
        balance = stock.balance_sheet.T
        balance = enrich(balance, t)
        balance.to_json(
            f"{BASE_PATH}/balance_sheet/{year}/{month}/{day}/{t}.json",
            orient="index"
        )
    except Exception as e:
        print("Balance error:", t, e)

    # ---------------- Cashflow ----------------
    try:
        cashflow = stock.cashflow.T
        cashflow = enrich(cashflow, t)
        cashflow.to_json(
            f"{BASE_PATH}/cashflow/{year}/{month}/{day}/{t}.json",
            orient="index"
        )
    except Exception as e:
        print("Cashflow error:", t, e)

    # ---------------- History ----------------
    try:
        history = stock.history(period="5y")
        history = enrich(history, t)
        history.to_json(
            f"{BASE_PATH}/history/{year}/{month}/{day}/{t}.json"
        )
    except Exception as e:
        print("History error:", t, e)

    # ---------------- Stats ----------------
    try:
        stats = pd.DataFrame([stock.info])
        stats = enrich(stats, t)
        stats.to_json(
            f"{BASE_PATH}/stats/{year}/{month}/{day}/{t}.json",
            orient="records"
        )
    except Exception as e:
        print("Stats error:", t, e)

print("\nYFinance ingestion complete")